In [1]:
import torch
import requests
from PIL import Image
from torchvision import transforms
from transformers import XLMRobertaTokenizer
import torch.nn.functional as F
import pandas as pd
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "4"  # GPU 사용 설정 (필요시 수정)

# BEiT-3 저장소의 필요한 클래스와 설정 함수를 직접 가져옵니다.
from modeling_finetune import BEiT3_Binary_Classification
from modeling_utils import _get_large_config

/data/2_data_server/cv-07/anaconda3/envs/beit3_new/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# 사용자 환경에 맞게 경로를 수정해주세요.
BASE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge"
MODEL_DIR = os.path.join(BASE_DIR, "finetuning/unilm/beit3/")
DATA_DIR = os.path.join(BASE_DIR, "data/data_2/")
# /data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/unilm/beit3/output_binary_classification/checkpoint-best.pth

TOKENIZER_PATH = os.path.join(BASE_DIR, "unilm/beit3/beit3.spm")
MODEL_WEIGHTS_PATH = os.path.join(MODEL_DIR, "output_binary_classification","checkpoint-best.pth")
TEST_CSV_PATH = os.path.join(DATA_DIR, "test.csv")
IMAGE_DIR = os.path.join(DATA_DIR, "test_input_images")
OUTPUT_CSV_PATH = os.path.join(DATA_DIR, "binary_auto.csv")


# 디바이스 설정 (GPU 우선 사용)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
# 토크나이저 로드
tokenizer = XLMRobertaTokenizer(TOKENIZER_PATH)
print("✅ Tokenizer 로드 완료")

✅ Tokenizer 로드 완료


In [8]:
args = _get_large_config(img_size=224)
args.vocab_size = 64010
# args.num_classes = 2  # BEiT-3는 다중 클래스 분류를 지원하므로, 이진 분류를 위해 num_classes를 2로 설정합니다.
print(f"✅ 모델 설정(args) 생성 완료 (img_size=224, vocab_size=64010)")


✅ 모델 설정(args) 생성 완료 (img_size=224, vocab_size=64010)


In [63]:
model = BEiT3_Binary_Classification(args=args)
print("✅ 모델 구조 생성 완료")

# 1. 'large' 모델 가중치를 로드할 때, weights_only=False 인자를 추가합니다.
state_dict = torch.load(MODEL_WEIGHTS_PATH, map_location='cpu', weights_only=False)["model"]

✅ 모델 구조 생성 완료


In [10]:
keys_to_delete = [key for key in state_dict if key.startswith('mim_head')]
for key in keys_to_delete:
    del state_dict[key]
print(f"✅ 불필요한 가중치 키 제거 완료: {keys_to_delete}")


model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()
print("✅ 모델 가중치 로드 및 초기화 완료")

✅ 불필요한 가중치 키 제거 완료: []
✅ 모델 가중치 로드 및 초기화 완료


In [11]:
# --- 2. 이미지 전처리 ---
transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
])

In [12]:
image_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test_input_images/TEST_000.jpg"

image = Image.open(image_path).convert("RGB")
image_tensor = transform(image).unsqueeze(0).to(device)

In [60]:
# What types of fruits are visible in the image?,Bananas and grapes placed in baskets,Apples and oranges displayed on the counter,Peaches and plums in a wooden crate,Pears and lemons arranged neatly
question = "What types of fruits are visible in the image?"
options = [
    "Bananas and grapes placed in baskets",
    "Apples and oranges displayed on the counter",
    "Peaches and plums in a wooden crate",
    "Pears and lemons arranged neatly"]

text_prompt = f"{question} {options[0]}"
encoding = tokenizer(text_prompt, return_tensors="pt")
language_tokens = encoding.input_ids.to(device)
padding_mask = encoding.attention_mask.to(device)


In [61]:
logits = model(
    image=image_tensor,
    text_description=language_tokens,
    padding_mask=padding_mask
).squeeze(-1)

print(f"✅ 로짓 계산 완료: {logits}")

✅ 로짓 계산 완료: tensor([2.6172], device='cuda:0', grad_fn=<SqueezeBackward1>)
